# 🧠 Sprint 3 — RAG Retrieval, Cohere Reranking & LangGraph Triage
**Project:** Autonomous Medical Research & Patient Triage Assistant
**Day:** 3 of 5 | **Sprint Goal:** Retrieve clinical evidence → Rerank → Classify urgency

## What we build today
```
Query Pinecone (top-10 clinical chunks)
  → Cohere Cross-Encoder Rerank (top-3 most relevant)
  → LangGraph Triage Agent:
       - Urgency level (Emergency / Urgent / Standard / Self-Care)
       - Differential diagnosis (top 3 conditions)
       - Recommended tests
       - Drug interaction check
       - Red flag confirmation
```

## Sprint 3 User Stories
> **US-03:** Retrieved chunks are reranked for relevance before analysis
> **US-04:** Clinician receives differential diagnosis with ICD-10 codes
> **US-05:** System retries failed calls and sends alerts on errors

**Definition of Done:**
- Cohere returns top-3 clinically relevant chunks
- LangGraph agent produces urgency_level, differential, urgency_score
- Error handler fires Telegram alert on failure

## Safety Rule
> Triage classification must always be grounded in retrieved evidence.
> The agent must NEVER invent clinical information.

In [ ]:
import os, json, time, uuid, pathlib
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from openai import OpenAI
from pinecone import Pinecone
import cohere

load_dotenv()
OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY', '')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY', '')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')

assert OPENAI_API_KEY,   'Set OPENAI_API_KEY'
assert PINECONE_API_KEY, 'Set PINECONE_API_KEY'
assert COHERE_API_KEY,   'Set COHERE_API_KEY'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
pc            = Pinecone(api_key=PINECONE_API_KEY)
cohere_client = cohere.Client(COHERE_API_KEY)

PINECONE_INDEX_NAME = 'medical-triage-agent'
EMBED_MODEL         = 'text-embedding-3-small'
TRIAGE_MODEL        = 'gpt-4o-mini'

pinecone_index = pc.Index(PINECONE_INDEX_NAME)
print('All clients initialised')
print(f'Pinecone index: {PINECONE_INDEX_NAME}')
print(f'Triage model  : {TRIAGE_MODEL}')

In [ ]:
class MedicalAgentState(TypedDict):
    session_id:        str
    ticket_id:         str
    initiated_at:      str
    channel:           str
    chat_id:           str
    user_mode:         str
    raw_message:       str
    symptoms:          List[str]
    duration:          Optional[str]
    age:               Optional[str]
    medications:       List[str]
    status:            str
    errors:            List[str]
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    clinical_namespace: Optional[str]
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    urgency_level:     Optional[str]
    urgency_score:     Optional[int]
    differential:      Optional[List[str]]
    red_flags:         Optional[List[str]]
    retry_count:       int
    triage_report:     Optional[Dict[str, str]]
    nutrition_advice:  Optional[str]
    home_remedies:     Optional[str]
    follow_up_sent:    bool
    report_ready:      bool
    workflow_path:     List[str]

print('MedicalAgentState schema ready')

In [ ]:
# Load state from Sprint 2 if available
if pathlib.Path('sprint2_medical_output.json').exists():
    data = json.loads(pathlib.Path('sprint2_medical_output.json').read_text())
    sprint2_state = MedicalAgentState(**{k: data.get(k) for k in MedicalAgentState.__annotations__})
    print(f'Loaded Sprint 2 state')
    print(f'  Symptoms    : {sprint2_state["symptoms"][:3]}')
    print(f'  Ticket      : {sprint2_state["ticket_id"]}')
    print(f'  Namespace   : {sprint2_state.get("clinical_namespace") or "(will be derived)"}')
    print(f'  Pinecone IDs: {len(sprint2_state.get("pinecone_ids") or [])} vectors')
else:
    # Create a test state if Sprint 2 was not run
    sprint2_state = MedicalAgentState(
        session_id=str(uuid.uuid4()),
        ticket_id=f'MED-{datetime.now(timezone.utc).strftime("%Y%m%d")}-TEST0001',
        initiated_at=datetime.now(timezone.utc).isoformat(),
        channel='telegram', chat_id='8138298582', user_mode='patient',
        raw_message='I have malaria fever chills headache for 3 days',
        symptoms=['malaria', 'fever', 'chills', 'headache'],
        duration='3 days', age=None, medications=[],
        status='stored', errors=[],
        raw_research=None, research_chunks=None, pinecone_ids=None,
        clinical_namespace=None,
        retrieved_chunks=None, reranked_chunks=None,
        urgency_level=None, urgency_score=None,
        differential=None, red_flags=[],
        retry_count=0, triage_report=None, nutrition_advice=None,
        home_remedies=None, follow_up_sent=False,
        report_ready=False, workflow_path=[]
    )
    print('No Sprint 2 output found - using test state')
    print('Make sure Sprint 2 ran first and Pinecone has data!')

In [ ]:
def retrieve_clinical_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Embed symptom query and retrieve top-10 clinical chunks from Pinecone.
    Uses the session-specific namespace created in Sprint 2 when available.
    """
    symptoms = state.get('symptoms', [])
    message  = state.get('raw_message', '')
    mode     = state.get('user_mode', 'patient')
    errors   = list(state.get('errors', []))
    namespace = state.get('clinical_namespace')
    if not namespace:
        condition_key = '-'.join(symptoms[0].split()[:3]).lower() if symptoms else 'general'
        condition_key = re.sub(r'[^a-z0-9-]', '', condition_key)[:50]
        ticket_suffix = state.get('ticket_id', 'UNKNOWN')[-8:].lower()
        namespace = f'{condition_key}-{ticket_suffix}'

    # Build clinical query
    symptom_str = ', '.join(symptoms[:5]) if symptoms else message[:200]
    if mode == 'clinician':
        query = f'Clinical differential diagnosis treatment protocol: {symptom_str}'
    else:
        query = f'Patient symptoms management guidelines Nigeria: {symptom_str}'

    print(f'[RETRIEVE] Query: {query[:70]}')

    fallback = [c for c in (state.get('research_chunks') or []) if isinstance(c, str) and c.strip()]
    for attempt in range(3):
        try:
            emb = openai_client.embeddings.create(model=EMBED_MODEL, input=[query], timeout=30)
            query_vector = emb.data[0].embedding

            # Query the matching Sprint 2 namespace for this session
            results = pinecone_index.query(
                vector=query_vector,
                top_k=5,
                include_metadata=True,
                namespace=namespace
            )

            chunks = [
                m['metadata']['text']
                for m in results['matches']
                if m.get('metadata', {}).get('text')
            ]

            if not chunks:
                if fallback:
                    return {
                        **state, 'retrieved_chunks': fallback[:10],
                        'status': 'retrieved_fallback',
                        'errors': errors + ['No clinical data in Pinecone. Used local research chunks fallback.'],
                        'workflow_path': state.get('workflow_path', []) + ['retrieve']
                    }
                return {
                    **state, 'status': 'error_no_clinical_data',
                    'errors': errors + ['No clinical data in Pinecone. Run Sprint 2 first.'],
                    'workflow_path': state.get('workflow_path', []) + ['retrieve']
                }

            # Log source breakdown
            sources = {}
            for m in results['matches']:
                src = m.get('metadata', {}).get('source_type', 'unknown')
                sources[src] = sources.get(src, 0) + 1

            print(f'[RETRIEVE] {len(chunks)} chunks | Sources: {sources}')

            return {
                **state, 'retrieved_chunks': chunks,
                'status': 'retrieved', 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['retrieve']
            }

        except Exception as e:
            print(f'  Retrieve attempt {attempt+1}/3 failed: {e}')
            if fallback:
                return {
                    **state, 'retrieved_chunks': fallback[:10],
                    'status': 'retrieved_fallback',
                    'errors': errors + [f'Pinecone retrieve failed: {e}', 'Used local research chunks fallback'],
                    'workflow_path': state.get('workflow_path', []) + ['retrieve']
                }
            if attempt == 2:
                return {
                    **state, 'status': 'error_retrieve_failed',
                    'errors': errors + [f'Pinecone retrieve failed: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['retrieve']
                }
            time.sleep(2 ** attempt)

print('retrieve_clinical_node defined')

In [ ]:
def rerank_clinical_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Rerank top-10 Pinecone chunks to top-3 using Cohere cross-encoder.
    Clinical reranking is critical — cosine similarity finds surface matches,
    Cohere finds true semantic clinical relevance.
    """
    symptoms = state.get('symptoms', [])
    chunks   = state.get('retrieved_chunks', [])
    mode     = state.get('user_mode', 'patient')
    errors   = list(state.get('errors', []))

    symptom_str = ', '.join(symptoms[:5])
    if mode == 'clinician':
        query = f'Clinical guidelines differential diagnosis treatment for: {symptom_str} in Nigerian healthcare setting'
    else:
        query = f'Health information management care guidance for patient with: {symptom_str} in Nigeria'

    print(f'[RERANK] Reranking {len(chunks)} chunks with Cohere...')

    for attempt in range(3):
        try:
            results = cohere_client.rerank(
                model='rerank-english-v3.0',
                query=query,
                documents=chunks,
                top_n=3,
                return_documents=True
            )
            reranked = [r.document.text for r in results.results if r.document]
            scores   = [round(r.relevance_score, 3) for r in results.results]
            print(f'[RERANK] Top-3 selected | Relevance scores: {scores}')
            return {
                **state, 'reranked_chunks': reranked,
                'status': 'reranked', 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['rerank']
            }
        except Exception as e:
            print(f'  Rerank attempt {attempt+1}/3 failed: {e}')
            if attempt == 2:
                print('[RERANK] Falling back to top-3 Pinecone results')
                return {
                    **state, 'reranked_chunks': chunks[:3],
                    'status': 'reranked_fallback',
                    'errors': errors + [f'Cohere rerank failed, used fallback: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['rerank']
                }
            time.sleep(2 ** attempt)

print('rerank_clinical_node defined')

In [ ]:
PATIENT_TRIAGE_PROMPT = """You are a medical triage AI assistant supporting Nigerian healthcare.
Analyse the patient symptoms using ONLY the provided clinical evidence context.
Return a JSON object with exactly these fields:

{
  "urgency_level": "emergency" | "urgent" | "standard" | "self-care",
  "urgency_score": <integer 1-10>,
  "differential": ["condition1", "condition2", "condition3"],
  "recommended_action": "<clear action for the patient>",
  "recommended_tests": ["test1", "test2"],
  "red_flags_present": ["flag1"] or [],
  "self_care_eligible": true | false,
  "facility_needed": "home" | "clinic" | "hospital" | "emergency_room",
  "confidence": "low" | "medium" | "high"
}

Urgency rules:
- emergency: life-threatening, go to A&E immediately
- urgent: see doctor within 24 hours
- standard: book appointment this week
- self-care: manage at home with guidance

IMPORTANT: Base ALL recommendations on the provided clinical context.
Return ONLY valid JSON. No markdown, no explanation."""

CLINICIAN_TRIAGE_PROMPT = """You are a clinical decision support AI for Nigerian healthcare providers.
Analyse the clinical presentation using the provided evidence context.
Return a JSON object with exactly these fields:

{
  "urgency_level": "emergency" | "urgent" | "standard" | "self-care",
  "urgency_score": <integer 1-10>,
  "differential": ["condition1 (ICD-10: X00)", "condition2 (ICD-10: X00)", "condition3 (ICD-10: X00)"],
  "recommended_action": "<clinical action>",
  "recommended_tests": ["test1", "test2", "test3"],
  "drug_recommendations": ["drug1 dose route", "drug2 dose route"],
  "drug_interactions": ["interaction1"] or [],
  "contraindications": ["contraindication1"] or [],
  "red_flags_present": ["flag1"] or [],
  "nhis_protocol": "<relevant NHIS Nigeria protocol if applicable>",
  "pubmed_evidence": "<cite any PubMed reference from context>",
  "confidence": "low" | "medium" | "high"
}

Include ICD-10 codes in differential diagnosis.
Return ONLY valid JSON. No markdown, no explanation."""

print('Triage prompts defined')
print(f'  Patient prompt  : {len(PATIENT_TRIAGE_PROMPT)} chars')
print(f'  Clinician prompt: {len(CLINICIAN_TRIAGE_PROMPT)} chars')

In [ ]:
def triage_agent_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: LangGraph triage agent - classifies urgency, differential diagnosis.
    Uses mode-specific prompts (patient vs clinician).
    """
    symptoms   = state.get('symptoms', [])
    message    = state.get('raw_message', '')
    chunks     = state.get('reranked_chunks') or state.get('retrieved_chunks') or []
    mode       = state.get('user_mode', 'patient')
    medications = state.get('medications', [])
    errors     = list(state.get('errors', []))

    context    = '\n\n---\n\n'.join(chunks)
    system_prompt = CLINICIAN_TRIAGE_PROMPT if mode == 'clinician' else PATIENT_TRIAGE_PROMPT

    user_content = (
        f'Clinical evidence context:\n{context}\n\n'
        f'Patient presentation:\n'
        f'Symptoms: {chr(44).join(symptoms)}\n'
        f'Full message: {message}\n'
        f'Current medications: {chr(44).join(medications) if medications else "None reported"}\n'
        f'Duration: {state.get("duration") or "Not specified"}\n'
        f'Age: {state.get("age") or "Not specified"}\n'
        f'Mode: {mode}\n\n'
        f'Classify urgency and provide triage assessment.'
    )

    print(f'[TRIAGE] Running LangGraph triage agent | Mode: {mode}')

    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=TRIAGE_MODEL, temperature=0,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_content}
                ]
            )
            parsed = json.loads(response.choices[0].message.content)

            urgency = parsed.get('urgency_level', 'standard')
            score   = int(parsed.get('urgency_score', 5))
            diff    = parsed.get('differential', [])

            print(f'[TRIAGE] Urgency: {urgency.upper()} ({score}/10)')
            print(f'[TRIAGE] Differential: {diff[:2]}')
            print(f'[TRIAGE] Action: {parsed.get("recommended_action", "")[:80]}')

            return {
                **state,
                'urgency_level':  urgency,
                'urgency_score':  score,
                'differential':   diff,
                'status':         'triaged',
                'errors':         errors,
                'triage_report':  {'_classification': json.dumps(parsed)},
                'workflow_path':  state.get('workflow_path', []) + ['triage_agent']
            }

        except Exception as e:
            print(f'  Triage attempt {attempt+1}/3 failed: {e}')
            if attempt == 2:
                return {
                    **state, 'status': 'error_triage_failed',
                    'errors': errors + [f'Triage agent failed: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['triage_agent']
                }
            time.sleep(2 ** attempt)

print('triage_agent_node defined')

In [ ]:
def drug_interaction_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Check drug interactions for medications mentioned by patient/clinician.
    Only runs if medications were detected in the input.
    """
    medications = state.get('medications', [])
    errors      = list(state.get('errors', []))

    if not medications:
        print('[DRUG CHECK] No medications reported - skipping')
        return {
            **state,
            'workflow_path': state.get('workflow_path', []) + ['drug_check_skipped']
        }

    print(f'[DRUG CHECK] Checking interactions for: {medications}')

    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=TRIAGE_MODEL, temperature=0,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': (
                        'You are a clinical pharmacist. Check drug interactions and contraindications.'
                        'Return JSON: {"interactions": [], "warnings": [], "safe_to_continue": true/false}'
                    )},
                    {'role': 'user', 'content': f'Check interactions for these medications: {medications}'}
                ]
            )
            parsed = json.loads(response.choices[0].message.content)
            interactions = parsed.get('interactions', [])

            if interactions:
                print(f'[DRUG CHECK] Interactions found: {interactions}')
            else:
                print('[DRUG CHECK] No significant interactions found')

            existing_report = state.get('triage_report') or {}
            existing_report['drug_interactions'] = json.dumps(parsed)

            return {
                **state, 'triage_report': existing_report, 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['drug_check']
            }

        except Exception as e:
            if attempt == 2:
                errors.append(f'Drug check failed: {e}')
                return {
                    **state, 'errors': errors,
                    'workflow_path': state.get('workflow_path', []) + ['drug_check_failed']
                }
            time.sleep(2 ** attempt)

print('drug_interaction_node defined')

In [ ]:
def error_handler_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    US-05: Error handler fires alert on any pipeline failure.
    In production: sends Telegram/WhatsApp alert to admin.
    """
    print('\n' + '!' * 55)
    print('  PIPELINE ERROR - ALERT FIRED')
    print('!' * 55)
    print(f'  Ticket  : {state.get("ticket_id")}')
    print(f'  Status  : {state.get("status")}')
    print(f'  Channel : {state.get("channel")}')
    for e in state.get('errors', []):
        print(f'  ERROR: {e}')
    print('  -> In production: Admin alert sent via Telegram')
    print('!' * 55)
    return {
        **state, 'status': 'failed',
        'workflow_path': state.get('workflow_path', []) + ['error_handler']
    }

print('error_handler_node defined (US-05 satisfied)')

In [ ]:
def route(state: MedicalAgentState) -> str:
    return 'error_handler' if 'error' in state.get('status', '') else 'continue'

def build_sprint3_graph():
    g = StateGraph(MedicalAgentState)
    g.add_node('retrieve',      retrieve_clinical_node)
    g.add_node('rerank',        rerank_clinical_node)
    g.add_node('triage_agent',  triage_agent_node)
    g.add_node('drug_check',    drug_interaction_node)
    g.add_node('error_handler', error_handler_node)

    g.set_entry_point('retrieve')
    g.add_conditional_edges('retrieve',     route, {'continue': 'rerank',       'error_handler': 'error_handler'})
    g.add_conditional_edges('rerank',       route, {'continue': 'triage_agent', 'error_handler': 'error_handler'})
    g.add_conditional_edges('triage_agent', route, {'continue': 'drug_check',   'error_handler': 'error_handler'})
    g.add_edge('drug_check',    END)
    g.add_edge('error_handler', END)
    return g.compile()

sprint3_graph = build_sprint3_graph()
print('Sprint 3 graph compiled')
print('Flow: retrieve -> rerank -> triage_agent -> drug_check -> END')

In [ ]:
print('=' * 60)
print(f'Running Sprint 3 for: {sprint2_state["raw_message"][:50]}')
print('=' * 60)

result = sprint3_graph.invoke(sprint2_state)

print(f'\n{"-" * 60}')
print(f'Status         : {result["status"]}')
print(f'Urgency Level  : {result.get("urgency_level")}')
print(f'Urgency Score  : {result.get("urgency_score")}/10')
print(f'Differential   : {(result.get("differential") or [])[:3]}')
print(f'Reranked chunks: {len(result.get("reranked_chunks") or [])}')
print(f'Workflow path  : {" -> ".join(result["workflow_path"])}')
print(f'Errors         : {result.get("errors", [])}')
print(f'{"-" * 60}')

In [ ]:
print('=' * 60)
print('SPRINT 3 - DEFINITION OF DONE CHECK')
print('=' * 60)

reranked_ok  = len(result.get('reranked_chunks') or []) == 3
urgency_ok   = result.get('urgency_level') in ('emergency', 'urgent', 'standard', 'self-care')
diff_ok      = len(result.get('differential') or []) >= 1
score_ok     = isinstance(result.get('urgency_score'), int)
path_ok      = 'triage_agent' in result.get('workflow_path', [])

print(f'{"OK" if reranked_ok else "FAIL"} Cohere returned top-3 chunks : {len(result.get("reranked_chunks") or [])} chunks')
print(f'{"OK" if urgency_ok  else "FAIL"} Urgency level classified     : {result.get("urgency_level")}')
print(f'{"OK" if diff_ok     else "FAIL"} Differential diagnosis       : {(result.get("differential") or [])[:2]}')
print(f'{"OK" if score_ok    else "FAIL"} Urgency score (1-10)         : {result.get("urgency_score")}')
print(f'{"OK" if path_ok     else "FAIL"} LangGraph triage node ran')

all_ok = all([reranked_ok, urgency_ok, diff_ok, score_ok, path_ok])
print(f'\n{"Sprint 3 DoD: MET" if all_ok else "Sprint 3 DoD: PARTIAL"}')
print('Ready for Sprint 4 - Report Generation')

# Save for Sprint 4
pathlib.Path('sprint3_medical_output.json').write_text(
    json.dumps(dict(result), indent=2, default=str))
print('Sprint 3 output saved to sprint3_medical_output.json')